# Create Car Drawer Implementation Guide

This notebook provides a comprehensive guide for implementing a 'Create Car' drawer with a form in a Next.js project. The drawer will be integrated with a speed dial component and include form validation, field population from the database, and submission handling.

## 1. Setup Drawer Component

Create a reusable drawer component `CarCreateDrawer.tsx` with a header, close button, and responsive layout.

**Key Features:**
- Header with title and close button
- Responsive layout that works on mobile and desktop
- Smooth open/close animations
- Overlay backdrop

In [ ]:
# CarCreateDrawer.tsx - Basic Structure
drawer_component = """
import React, { useState } from 'react';
import { X } from 'lucide-react';

interface CarCreateDrawerProps {
  isOpen: boolean;
  onClose: () => void;
}

export const CarCreateDrawer: React.FC<CarCreateDrawerProps> = ({ isOpen, onClose }) => {
  return (
    <>
      {/* Backdrop Overlay */}
      {isOpen && (
        <div 
          className="fixed inset-0 bg-black bg-opacity-50 z-40"
          onClick={onClose}
        />
      )}
      
      {/* Drawer Container */}
      <div 
        className={`fixed right-0 top-0 h-full w-full max-w-md bg-white shadow-lg transform transition-transform duration-300 ease-in-out z-50 ${
          isOpen ? 'translate-x-0' : 'translate-x-full'
        }`}
      >
        {/* Header */}
        <div className="flex items-center justify-between p-6 border-b">
          <h2 className="text-xl font-semibold text-gray-900">Create New Car</h2>
          <button
            onClick={onClose}
            className="p-1 hover:bg-gray-100 rounded-md transition-colors"
          >
            <X className="w-5 h-5" />
          </button>
        </div>
        
        {/* Content Area */}
        <div className="overflow-y-auto h-[calc(100vh-80px)]">
          {/* Form will go here */}
        </div>
      </div>
    </>
  );
};
"""

print(drawer_component)

## 2. Create Form Layout

Design the form layout using Tailwind and shadcn-style components, grouping fields into sections like 'Basic Information', 'Car Details', and 'Remarks'.

**Layout Structure:**
- Basic Information Section (Vehicle Type, Brand)
- Car Details Section (Model, Year, Transmission, etc.)
- Remarks Section (Additional notes)
- Action Buttons (Submit, Cancel)

In [ ]:
# Form Layout Structure
form_layout = """
interface FormSection {
  title: string;
  description?: string;
  fields: FormField[];
}

const formSections: FormSection[] = [
  {
    title: 'Basic Information',
    description: 'Essential details about the vehicle',
    fields: ['vehicleTypeId', 'brandId']
  },
  {
    title: 'Car Details',
    description: 'Specific information about this car',
    fields: ['model', 'year', 'transmission', 'fuelType', 'color', 'licensePlate', 'mileage']
  },
  {
    title: 'Additional Information',
    description: 'Extra details and remarks',
    fields: ['remarks']
  }
];
"""

print(form_layout)

## 3. Implement Form Fields

Add all required fields with appropriate input types, labels, placeholders, and helper texts. This includes select fields, text inputs, number inputs, and textarea.

In [ ]:
# Form Fields Implementation
form_fields = """
// Define all form fields with their properties
const formFieldsConfig = {
  vehicleTypeId: {
    label: 'Vehicle Type',
    type: 'select',
    placeholder: 'Select vehicle type',
    required: true,
    helperText: 'Choose the type of vehicle'
  },
  brandId: {
    label: 'Brand',
    type: 'select',
    placeholder: 'Select brand',
    required: true,
    helperText: 'Choose the vehicle brand'
  },
  model: {
    label: 'Model',
    type: 'text',
    placeholder: 'e.g., Civic, Corolla',
    required: true,
    helperText: 'Enter the model name'
  },
  year: {
    label: 'Year',
    type: 'number',
    placeholder: 'e.g., 2023',
    required: true,
    min: 1990,
    max: new Date().getFullYear() + 1,
    helperText: 'Manufacturing year'
  },
  transmission: {
    label: 'Transmission',
    type: 'select',
    placeholder: 'Select transmission type',
    required: true,
    options: ['Automatic', 'Manual'],
    helperText: 'Choose transmission type'
  },
  fuelType: {
    label: 'Fuel Type',
    type: 'select',
    placeholder: 'Select fuel type',
    required: true,
    options: ['Petrol', 'Diesel', 'Hybrid', 'Electric'],
    helperText: 'Choose fuel type'
  },
  color: {
    label: 'Color',
    type: 'text',
    placeholder: 'e.g., Red, Blue, Black',
    required: true,
    helperText: 'Vehicle color'
  },
  licensePlate: {
    label: 'License Plate',
    type: 'text',
    placeholder: 'e.g., ABC-1234',
    required: true,
    helperText: 'Vehicle registration number'
  },
  mileage: {
    label: 'Mileage (km)',
    type: 'number',
    placeholder: '0',
    required: true,
    min: 0,
    helperText: 'Current mileage in kilometers'
  },
  remarks: {
    label: 'Remarks',
    type: 'textarea',
    placeholder: 'Add any additional notes...',
    required: false,
    helperText: 'Optional remarks about the vehicle',
    rows: 4
  }
};
"""

print(form_fields)

## 4. Add Validation with Zod

Define a Zod schema for form validation, ensuring required fields, proper formats, and constraints like mileage >= 0.

In [ ]:
# Zod Validation Schema
zod_schema = """
import { z } from 'zod';

export const createCarSchema = z.object({
  vehicleTypeId: z.string().min(1, 'Vehicle type is required'),
  brandId: z.string().min(1, 'Brand is required'),
  model: z.string()
    .min(1, 'Model is required')
    .min(2, 'Model must be at least 2 characters')
    .max(50, 'Model must not exceed 50 characters'),
  year: z.number()
    .int('Year must be an integer')
    .min(1990, 'Year must be 1990 or later')
    .max(new Date().getFullYear() + 1, 'Year cannot be in the future'),
  transmission: z.enum(['Automatic', 'Manual'], {
    errorMap: () => ({ message: 'Select a valid transmission type' })
  }),
  fuelType: z.enum(['Petrol', 'Diesel', 'Hybrid', 'Electric'], {
    errorMap: () => ({ message: 'Select a valid fuel type' })
  }),
  color: z.string()
    .min(1, 'Color is required')
    .min(2, 'Color must be at least 2 characters')
    .max(30, 'Color must not exceed 30 characters'),
  licensePlate: z.string()
    .min(1, 'License plate is required')
    .regex(/^[A-Z0-9\-]{5,}$/, 'Invalid license plate format'),
  mileage: z.number()
    .int('Mileage must be an integer')
    .min(0, 'Mileage cannot be negative'),
  remarks: z.string().optional().default('')
});

export type CreateCarInput = z.infer<typeof createCarSchema>;
"""

print(zod_schema)

## 5. Handle Form Submission

Implement a placeholder onSubmit handler that logs form data and prepares for API integration. This includes error handling and loading states.

In [ ]:
# Form Submission Handler
form_submission = """
import { useForm } from 'react-hook-form';
import { zodResolver } from '@hookform/resolvers/zod';
import { createCarSchema, CreateCarInput } from '@/lib/schemas';

const handleSubmit = async (data: CreateCarInput) => {
  try {
    setIsLoading(true);
    
    // Log the form data
    console.log('Creating car with data:', data);
    
    // API Integration - Replace with actual endpoint
    const response = await fetch('/api/cars', {
      method: 'POST',
      headers: {
        'Content-Type': 'application/json',
      },
      body: JSON.stringify(data),
    });
    
    if (!response.ok) {
      throw new Error('Failed to create car');
    }
    
    const result = await response.json();
    console.log('Car created successfully:', result);
    
    // Show success message
    toast.success('Car created successfully');
    
    // Reset form and close drawer
    reset();
    onClose();
    
    // Refresh car list
    router.refresh();
  } catch (error) {
    console.error('Error creating car:', error);
    toast.error(error instanceof Error ? error.message : 'Failed to create car');
  } finally {
    setIsLoading(false);
  }
};
"""

print(form_submission)

## 6. Integrate Drawer with Speed Dial

Connect the drawer to the speed dial on the `/cars` page, ensuring it opens when 'Create Car' is clicked. The speed dial should trigger the drawer state.

In [ ]:
# Speed Dial Integration
speed_dial_integration = """
// In the /cars page component

import { useState } from 'react';
import { CarCreateDrawer } from '@/components/cars/CarCreateDrawer';
import { SpeedDial } from '@/components/ui/SpeedDial';

export default function CarsPage() {
  const [isDrawerOpen, setIsDrawerOpen] = useState(false);
  
  const handleCreateCar = () => {
    setIsDrawerOpen(true);
  };
  
  const handleCloseDrawer = () => {
    setIsDrawerOpen(false);
  };
  
  return (
    <div className="space-y-6">
      {/* Page Header and Content */}
      <div className="space-y-4">
        <h1 className="text-3xl font-bold">Cars</h1>
        {/* Cars list/table will go here */}
      </div>
      
      {/* Speed Dial Button */}
      <SpeedDial 
        onCreateCar={handleCreateCar}
      />
      
      {/* Create Car Drawer */}
      <CarCreateDrawer 
        isOpen={isDrawerOpen}
        onClose={handleCloseDrawer}
      />
    </div>
  );
}
"""

print(speed_dial_integration)

## 7. Fetch Data for Select Fields

Fetch `vehicle_type` and `brand` data from Prisma and populate the select fields, handling empty states gracefully.

In [ ]:
# Data Fetching and Population
data_fetching = """
// API Route: /api/vehicle-types
import { prisma } from '@/lib/prisma';

export async function GET(request: Request) {
  try {
    const vehicleTypes = await prisma.vehicle_type.findMany({
      orderBy: { name: 'asc' },
      select: { id: true, name: true }
    });
    
    if (vehicleTypes.length === 0) {
      return Response.json({ 
        error: 'No vehicle types available' 
      }, { status: 404 });
    }
    
    return Response.json(vehicleTypes);
  } catch (error) {
    console.error('Error fetching vehicle types:', error);
    return Response.json({ 
      error: 'Failed to fetch vehicle types' 
    }, { status: 500 });
  }
}

// API Route: /api/brands
export async function GET(request: Request) {
  try {
    const brands = await prisma.brand.findMany({
      orderBy: { name: 'asc' },
      select: { id: true, name: true }
    });
    
    if (brands.length === 0) {
      return Response.json({ 
        error: 'No brands available' 
      }, { status: 404 });
    }
    
    return Response.json(brands);
  } catch (error) {
    console.error('Error fetching brands:', error);
    return Response.json({ 
      error: 'Failed to fetch brands' 
    }, { status: 500 });
  }
}

// Component: Hook to fetch select options
import { useEffect, useState } from 'react';

export function useSelectOptions(endpoint: string) {
  const [options, setOptions] = useState([]);
  const [isLoading, setIsLoading] = useState(true);
  const [error, setError] = useState<string | null>(null);
  
  useEffect(() => {
    const fetchOptions = async () => {
      try {
        setIsLoading(true);
        const response = await fetch(endpoint);
        
        if (!response.ok) {
          throw new Error('Failed to fetch options');
        }
        
        const data = await response.json();
        setOptions(data);
        setError(null);
      } catch (err) {
        setError(err instanceof Error ? err.message : 'Unknown error');
        setOptions([]);
      } finally {
        setIsLoading(false);
      }
    };
    
    fetchOptions();
  }, [endpoint]);
  
  return { options, isLoading, error };
}
"""

print(data_fetching)

## Implementation Summary

### Key Components to Create:
1. **CarCreateDrawer.tsx** - Main drawer component with layout
2. **CarCreateForm.tsx** - Form component with validation and submission
3. **useSelectOptions.ts** - Custom hook for fetching select field data
4. **createCarSchema.ts** - Zod validation schema
5. **API Routes** - `/api/cars`, `/api/vehicle-types`, `/api/brands`

### Integration Steps:
1. Create the drawer component with responsive design
2. Build the form with all required fields
3. Add Zod validation schema
4. Implement API endpoints for data fetching
5. Connect to speed dial on `/cars` page
6. Add success/error handling with toast notifications

### Best Practices:
- Use TypeScript for type safety
- Implement proper error handling
- Add loading states during submission
- Show validation errors inline
- Reset form after successful submission
- Use React Hook Form with Zod for validation
- Implement optimistic UI updates where possible

## Implementation Summary

### Key Components to Create:
1. **CarCreateDrawer.tsx** - Main drawer component with layout
2. **CarCreateForm.tsx** - Form component with validation and submission
3. **useSelectOptions.ts** - Custom hook for fetching select field data
4. **createCarSchema.ts** - Zod validation schema
5. **API Routes** - `/api/cars`, `/api/vehicle-types`, `/api/brands`

### Integration Steps:
1. Create the drawer component with responsive design
2. Build the form with all required fields
3. Add Zod validation schema
4. Implement API endpoints for data fetching
5. Connect to speed dial on `/cars` page
6. Add success/error handling with toast notifications

### Best Practices:
- Use TypeScript for type safety
- Implement proper error handling
- Add loading states during submission
- Show validation errors inline
- Reset form after successful submission
- Use React Hook Form with Zod for validation
- Implement optimistic UI updates where possible